# Football Player/Referee ReID Training

Fine-tunes a small **OSNet-x0.25** re-identification embedding model on
SoccerNet-ReID crops. The resulting weights are meant to plug into a
persistent "lost-track gallery" stage in the tracking pipeline (see the
project's tracking-limitations and robust-structure reports) so a player,
goalkeeper, or referee keeps their identity after leaving and re-entering
the frame, instead of the tracker minting a new ID.

This notebook does **not** retrain the detector -- see the companion
`notebooks/detection_model_training_v2.ipynb` for that.

This notebook is designed to run end-to-end **without unhandled errors**:
every external dependency (GPU, credentials, dataset) is checked explicitly
and fails with a clear instruction rather than a raw traceback if missing.

## 0) Prerequisite -- read this before running anything below

**SoccerNet-ReID still needs a one-time human step you must do outside this
notebook**, but it's much smaller than before:

1. Register at https://www.soccer-net.org/data and accept the NDA/EULA.
   You'll receive a password by email.
2. Add that password as a Kaggle **Secret** named `SOCCERNET_PASSWORD`
   (Add-ons -> Secrets) -- the same secret the detector/GSR notebooks use.

That's it. **Downloading, extracting, and preparing the data now happens
automatically inside this notebook** (Steps 2-3 below), the same pattern
`detection_model_training_v2.ipynb` and `gsr_minimap_model_training.ipynb`
already use -- no separate throwaway notebook, no manual Kaggle Dataset
upload, no "Add Data" step required.

If you already have a SoccerNet-ReID (or similarly-shaped) dataset attached
manually via **Add Data**, this notebook will find and use that first and
skip the automatic download -- see Step 2.

## 1) GPU check

In [47]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Go to Notebook Settings -> Accelerator and "
        "select a GPU (T4 x2 or P100), then Save & re-run this cell."
    )

print("GPU OK:", torch.cuda.get_device_name(0))

GPU OK: Tesla T4


## 1b) Credentials (Kaggle Secret, with manual fallback)

Never hardcode the real password directly in a notebook cell body.

In [48]:
SOCCERNET_PASSWORD = ""  # fallback: paste your SoccerNet NDA password here ONLY if not using Secrets

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try:
        SOCCERNET_PASSWORD = _secrets.get_secret("SOCCERNET_PASSWORD")
    except Exception:
        pass
except Exception:
    print("kaggle_secrets not available (not running on Kaggle?) -- using the "
          "manual fallback variable above.")

print("Credentials loaded (value not printed)." if SOCCERNET_PASSWORD else
      "WARNING: SOCCERNET_PASSWORD is empty -- the auto-download in Step 2 "
      "will be skipped unless a dataset is already manually attached.")

Credentials loaded (value not printed).


## 2) Get the ReID dataset -- manually-attached first, else auto-download

1. First checks a few common manually-attached **Add Data** mount paths
   (in case you already prepared a Kaggle Dataset the old way -- still
   supported).
2. If none are found and `SOCCERNET_PASSWORD` is set, downloads
   SoccerNet-ReID directly via `SoccerNetDownloader`, the same defensive,
   task-name-fallback pattern used in the other two notebooks.

**Correction from an earlier version of this notebook:** the `reid` task is
NOT small -- a real run showed it's ~14.5GB (delivered via the Hugging Face
Hub, not a small direct zip), which blows Kaggle's ~20GB `/kaggle/working`
quota if fully extracted *and* then copied again into the training layout
(what the old Step 4 did). This version never extracts the archives at all:
Step 4 streams identity crops directly out of the zip members straight into
the final training layout, and deletes each zip right after it's consumed --
the same streaming approach already used for SoccerNet-Tracking in
`detection_model_training_v2.ipynb`, applied here for the same disk-budget
reason.

In [49]:
from pathlib import Path
import shutil as _shutil

# 2a) Manually-attached dataset check (backward-compatible with the old
# workflow -- try a couple of plausible slugs).
_manual_candidates = [
    Path("/kaggle/input/soccernet-reid"),
    Path("/kaggle/input/soccernet-re-id"),
]
REID_DATA_ROOT = None
for _c in _manual_candidates:
    if _c.is_dir() and any(_c.iterdir()):
        REID_DATA_ROOT = _c
        print(f"Found manually-attached ReID dataset at: {REID_DATA_ROOT}")
        break

SOCCERNET_DIR = Path("/kaggle/working/SoccerNetReID")

# 2b) Auto-download if nothing manually attached.
if REID_DATA_ROOT is None:
    if not SOCCERNET_PASSWORD:
        raise RuntimeError(
            "No manually-attached ReID dataset found, and SOCCERNET_PASSWORD "
            "is empty so auto-download can't run either.\n\n"
            "Register at https://www.soccer-net.org/data, add the emailed "
            "password as a Kaggle Secret named SOCCERNET_PASSWORD, then "
            "re-run from Step 1b."
        )

    # Clean up any partial state from a previous failed attempt (e.g. an
    # earlier run that hit "No space left on device" mid-extraction) so
    # retries start from a known-clean disk instead of accumulating cruft.
    for _stale in (SOCCERNET_DIR, Path("/kaggle/working/reid_market1501"),
                   Path.home() / ".cache" / "huggingface"):
        if _stale.exists():
            _shutil.rmtree(_stale, ignore_errors=True)
            print(f"Cleaned up stale {_stale}")

    !pip install -q "SoccerNet>=0.1.61"
    from SoccerNet.Downloader import SoccerNetDownloader

    downloader = SoccerNetDownloader(LocalDirectory=str(SOCCERNET_DIR))
    downloader.password = SOCCERNET_PASSWORD

    _last_exc = None
    _downloaded = False
    for _task in ("reid", "tracking"):
        try:
            downloader.downloadDataTask(task=_task, split=["train", "test"])
            print(f"SoccerNet task '{_task}' downloaded successfully.")
            _downloaded = True
            _last_exc = None
            break
        except Exception as _exc:
            _last_exc = _exc
            print(f"SoccerNet task '{_task}' failed ({_exc!r}), trying next...")
    if not _downloaded:
        raise RuntimeError(
            "SoccerNet-ReID auto-download failed for every task name tried "
            f"({_last_exc!r}). Check https://www.soccer-net.org/data for the "
            "current ReID task name and update the _task tuple above, or "
            "fall back to manually attaching a prepared dataset via Add Data."
        )

    # Deliberately NOT extracting here -- Step 4 streams straight out of
    # these zip members into the final training layout, so the archives
    # never need to exist unpacked on disk at all (see the note above).
    REID_DATA_ROOT = SOCCERNET_DIR

if not REID_DATA_ROOT.is_dir() or not any(REID_DATA_ROOT.iterdir()):
    raise RuntimeError(
        f"REID_DATA_ROOT ({REID_DATA_ROOT}) is missing or empty after "
        "download -- inspect /kaggle/working/SoccerNetReID manually to see "
        "what actually landed there."
    )

print("Using ReID dataset at:", REID_DATA_ROOT)
_zips = list(REID_DATA_ROOT.rglob("*.zip"))
_dirs = [p for p in REID_DATA_ROOT.rglob("*") if p.is_dir()]
print(f"Found {len(_zips)} zip archive(s): {[z.name for z in _zips]}")
for _z in _zips:
    print(f"  {_z}: {_z.stat().st_size / 1e9:.2f} GB (not extracted)")


Cleaned up stale /kaggle/working/SoccerNetReID
Cleaned up stale /kaggle/working/reid_market1501
Cleaned up stale /root/.cache/huggingface


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

SoccerNet task 'reid' downloaded successfully.
Using ReID dataset at: /kaggle/working/SoccerNetReID
Found 2 zip archive(s): ['test.zip', 'train.zip']
  /kaggle/working/SoccerNetReID/reid/test.zip: 2.41 GB (not extracted)
  /kaggle/working/SoccerNetReID/reid/train.zip: 12.11 GB (not extracted)


## 3) Install pinned dependencies

In [50]:
!pip install -q "torchreid==0.2.5"

try:
    import torchreid  # noqa: F401
    print("torchreid import OK")
except Exception as exc:
    raise RuntimeError(
        f"torchreid failed to import after install ({exc!r}). This package's "
        "dependency footprint can be fragile on some Kaggle base images. Try "
        "restarting the kernel and re-running from Step 3, or pin a "
        "different torchreid/torch combination."
    ) from exc

torchreid import OK


## 4) Stream SoccerNet-ReID crops straight into a Market1501-style layout

Reads identity crops directly out of the (still-zipped) `train`/`test`
archives from Step 2 and writes only the sampled subset straight into the
final `torchreid`-ready layout -- the archives are never fully extracted,
and each is deleted right after being consumed to keep peak disk usage
bounded regardless of how large the full dataset is.

**Correction from an earlier version, after a real training run:** the
first identity-grouping heuristic (parent-folder name alone, e.g. `"20"`)
was a real bug -- those are per-match jersey/action numbers, reused in
every match, so it silently merged different real players across
different games under one "identity". The training run's symptom matched
this exactly: training accuracy climbed to 86% while rank-1 on held-out
identities stayed at 30% (memorizing corrupted labels, not learning real
identity). `_identity_key` now also incorporates the match folder, so
identities are only merged within the same game.

Also raised `MAX_IMAGES_PER_IDENTITY` (disk headroom was proven fine) and
added a minimum-crop-size filter -- SoccerNet's own filenames encode the
crop's pixel dimensions (e.g. `...-130x63.png`), so this is a free,
decode-free check.

**Important:** the match-folder-qualification heuristic (which path
segment counts as "the match") is still a best-effort guess from the
sample paths seen so far -- print the identity count after re-running and
sanity check it against the earlier (bugged) count of ~60; a large jump is
the expected signal this actually fixed the merge bug.

**Second correction, after another real run:** even with match-qualified
identity *keys*, a real run still came back with only 53 train identities
for ~9,000 real ones -- because `torchreid`'s Market1501 loader parses the
person id out of the *filename* with its own regex (digits/hyphens
immediately before `_c`), not from our Python-side grouping. That regex
silently truncated `"<match>__<jersey>"` down to just the trailing jersey
number, re-merging everything again. Fixed by mapping each real identity to
a clean sequential integer before building the filename, so the whole
identity survives into what torchreid actually parses.

In [51]:
import re
import shutil
import zipfile
from collections import defaultdict

REID_ROOT = Path("/kaggle/working/reid_market1501")
# Clean up any partial output from a previous crashed attempt (e.g. the
# "No space left on device" case) before starting fresh -- otherwise a
# retry just resumes into an already-full disk.
if REID_ROOT.exists():
    shutil.rmtree(REID_ROOT, ignore_errors=True)
    print(f"Cleaned up partial {REID_ROOT} from a previous attempt.")
(REID_ROOT / "bounding_box_train").mkdir(parents=True, exist_ok=True)
(REID_ROOT / "bounding_box_test").mkdir(parents=True, exist_ok=True)
(REID_ROOT / "query").mkdir(parents=True, exist_ok=True)

# The match-qualified identity fix (see _identity_key below) correctly
# revealed ~9,000 real identities in train.zip alone -- far more than the
# old (buggy) ~60. That's great for training diversity, but at
# MAX_IMAGES_PER_IDENTITY=50 it tried to write far more data than fits on
# Kaggle's disk, and crashed. Two changes fix this: fewer images/identity
# (breadth now matters more than depth, with this many real identities),
# and a hard cap on how many identities get used per split, evenly
# subsampled across the full range so match diversity is preserved rather
# than only keeping whichever matches happen to sort first.
MAX_IMAGES_PER_IDENTITY = 8
MAX_IDENTITIES_PER_SPLIT = 1200
MIN_CROP_DIM = 40              # px; skip crops smaller than this on either side (mostly noise)
IMAGE_EXTS = (".jpg", ".jpeg", ".png")
_DISK_SAFETY_FREE_BYTES = 3 * 1024**3  # stop early if free space drops below this


def _identity_key(member_path: str) -> str:
    """Best-effort identity grouping for one zip member path.

    IMPORTANT FIX: the jersey/action-number folder alone (e.g. "20") is
    NOT globally unique -- it's reused in every match, so grouping by it
    alone silently merges different real players across different games.
    This qualifies the identity with the match folder too (the path
    segment identifying the specific game, which the sample paths showed
    sits a few levels above the jersey-number folder), so identities are
    only merged within the same match.

    Falls back to the filename's leading digit run (common
    "<id>_<frame>.jpg"-style flat layouts) if there's no useful directory
    structure. Verify the resulting identity count against the printed
    sample paths below -- it should be substantially higher than the
    previous (bugged) ~60/56 if this is working correctly.
    """
    parts = member_path.strip("/").split("/")
    if len(parts) >= 2:
        jersey_folder = parts[-2]
        # The match folder is the parent of the jersey-number folder;
        # walk up a bit further if that segment looks too generic
        # (e.g. "train", "gallery") to actually identify one game.
        match_folder = parts[-3] if len(parts) >= 3 else ""
        for seg in reversed(parts[:-2]):
            if seg.lower() not in ("train", "test", "gallery", "query", "reid"):
                match_folder = seg
                break
        return f"{match_folder}__{jersey_folder}" if match_folder else jersey_folder
    match = re.match(r"(\d+)", Path(member_path).stem)
    return match.group(1) if match else Path(member_path).stem


def _crop_dims(member_path: str):
    """Parses trailing '-<W>x<H>' from a SoccerNet-ReID filename, if
    present (SoccerNet encodes the crop's pixel dimensions there). Returns
    None if it can't be parsed -- callers should not filter in that case."""
    m = re.search(r"-(\d+)x(\d+)\.\w+$", member_path)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))


def _stream_split(zip_path: Path, dst_dirs, cam_alternate=False):
    """Streams one zip's crops into dst_dirs (one target folder, or a
    [primary, secondary] pair to alternate into e.g. query/bounding_box_test).
    Deletes the zip afterward. Returns (n_identities, n_images)."""
    if not zip_path.exists():
        return 0, 0
    n_ids, n_imgs = 0, 0
    with zipfile.ZipFile(zip_path) as zf:
        by_id = defaultdict(list)
        for name in zf.namelist():
            if name.endswith("/") or not name.lower().endswith(IMAGE_EXTS):
                continue
            by_id[_identity_key(name)].append(name)

        n_skipped_small = 0
        for pid in list(by_id):
            kept = []
            for member in by_id[pid]:
                dims = _crop_dims(member)
                if dims is not None and (dims[0] < MIN_CROP_DIM or dims[1] < MIN_CROP_DIM):
                    n_skipped_small += 1
                    continue
                kept.append(member)
            if kept:
                by_id[pid] = kept
            else:
                del by_id[pid]

        all_ids = sorted(by_id)  # deterministic order for even subsampling
        if len(all_ids) > MAX_IDENTITIES_PER_SPLIT:
            stride = len(all_ids) / MAX_IDENTITIES_PER_SPLIT
            selected = [all_ids[int(i * stride)] for i in range(MAX_IDENTITIES_PER_SPLIT)]
            print(f"  {zip_path.name}: {len(by_id)} identities found "
                  f"({n_skipped_small} crops skipped as smaller than {MIN_CROP_DIM}px) "
                  f"-- subsampled to {len(selected)} (evenly spread for match diversity)")
        else:
            selected = all_ids
            print(f"  {zip_path.name}: {len(by_id)} identities found "
                  f"({n_skipped_small} crops skipped as smaller than {MIN_CROP_DIM}px)")

        for pid in selected[:3]:
            print(f"    sample id={pid!r}: {by_id[pid][:2]}")

        # CRITICAL: torchreid's Market1501 loader parses the person id out
        # of the filename with a regex that only captures the run of
        # digits/hyphens immediately before "_c" -- it does NOT treat the
        # filename as an opaque string. Our match-qualified identity keys
        # (e.g. "2015-11-07 ... Toulouse__20") get silently truncated by
        # that regex down to just the trailing "20", which re-merges
        # different real players across different matches right back
        # together -- the exact bug this was supposed to fix (confirmed by
        # a real run: only 53 "identities" came out of ~9,000 real ones).
        # Fix: map each real identity to a clean sequential integer BEFORE
        # building the filename, so the regex captures the whole thing.
        numeric_id_of = {pid: idx for idx, pid in enumerate(selected)}
        print(f"    Assigned {len(numeric_id_of)} numeric ids "
              f"(e.g. {selected[0]!r} -> {numeric_id_of[selected[0]]})")

        stopped_early = False
        for i, pid in enumerate(selected):
            numeric_id = numeric_id_of[pid]
            if i % 200 == 0:
                free = shutil.disk_usage("/kaggle/working").free
                if free < _DISK_SAFETY_FREE_BYTES:
                    print(f"    Stopping early at {i}/{len(selected)} identities -- "
                          f"only {free / 1e9:.2f} GB free, below the "
                          f"{_DISK_SAFETY_FREE_BYTES / 1e9:.0f} GB safety margin. "
                          "This is a graceful stop, not an error -- what's already "
                          "streamed is still usable for training.")
                    stopped_early = True
                    break
            members = by_id[pid]
            n_ids += 1
            for cam_idx, member in enumerate(members[:MAX_IMAGES_PER_IDENTITY]):
                dst_dir = (dst_dirs[0] if not cam_alternate or cam_idx > 0
                           else dst_dirs[-1]) if isinstance(dst_dirs, (list, tuple)) else dst_dirs
                # torchreid's built-in Market1501 loader only globs *.jpg
                # internally -- our crops are natively .png (SoccerNet's
                # format). PIL opens by file *content*, not extension, so
                # forcing a .jpg name here is safe (no re-encoding needed)
                # and is what actually satisfies that glob.
                fname = f"{numeric_id:06d}_c{cam_idx % 6 + 1}s1_{i:04d}_{cam_idx:02d}.jpg"
                with zf.open(member) as src, open(dst_dir / fname, "wb") as out:
                    out.write(src.read())
                n_imgs += 1
            if (i + 1) % 100 == 0:
                print(f"    ...{i + 1}/{len(selected)} identities streamed")

        if stopped_early:
            # Don't try to delete the zip -- we didn't finish reading it,
            # and there may be nothing to gain from unlinking it now anyway
            # (disk is already at the safety margin).
            print(f"  Kept {zip_path.name} (stopped early; re-run to resume "
                  "using more disk if you free some up, or accept this partial set).")
            return n_ids, n_imgs

    zip_path.unlink(missing_ok=True)
    print(f"  Removed {zip_path.name} after streaming ({n_ids} ids, {n_imgs} images).")
    return n_ids, n_imgs


train_zip = REID_DATA_ROOT / "reid" / "train.zip"
test_zip = REID_DATA_ROOT / "reid" / "test.zip"
if not train_zip.exists():
    # Fall back to whatever .zip files were actually found, in case the
    # internal path layout differs from the reid/<split>.zip assumption.
    _all_zips = sorted(REID_DATA_ROOT.rglob("*.zip"))
    train_zip = next((z for z in _all_zips if "train" in z.name.lower()), None)
    test_zip = next((z for z in _all_zips if "test" in z.name.lower()), None)

n_train_ids = n_train_imgs = 0
if train_zip and train_zip.exists():
    n_train_ids, n_train_imgs = _stream_split(
        train_zip, REID_ROOT / "bounding_box_train"
    )
else:
    print("No train.zip found -- check Step 2's zip listing above.")

n_test_ids = n_test_imgs = 0
if test_zip and test_zip.exists():
    n_test_ids, n_test_imgs = _stream_split(
        test_zip, [REID_ROOT / "bounding_box_test", REID_ROOT / "query"],
        cam_alternate=True,
    )
else:
    print("No test.zip found -- check Step 2's zip listing above.")

print(f"\nTrain: {n_train_ids} identities, {n_train_imgs} images")
print(f"Test/query: {n_test_ids} identities, {n_test_imgs} images")

assert n_train_imgs > 0 and n_test_imgs > 0, (
    "Streaming produced an empty split -- inspect the sample member paths "
    "printed above (or re-run with a debug print of zf.namelist()[:20] on "
    "one zip) and fix `_identity_key` to match the real internal layout."
)


  train.zip: 9188 identities found (44931 crops skipped as smaller than 40px) -- subsampled to 1200 (evenly spread for match diversity)
    sample id='2014-11-04 - 22-45 Arsenal 3 - 3 Anderlecht__0': ['train/europe_uefa-champions-league/2014-2015/2014-11-04 - 22-45 Arsenal 3 - 3 Anderlecht/0/53051-1892-32539-4959-Goalkeeper_team_left-1-03aa00000017460b000d-88x53.png', 'train/europe_uefa-champions-league/2014-2015/2014-11-04 - 22-45 Arsenal 3 - 3 Anderlecht/0/53032-1892-32538-4957-Player_team_right-None-03ar000_105173ab000e-124x50.png']
    sample id='2014-11-04 - 22-45 Arsenal 3 - 3 Anderlecht__15': ['train/europe_uefa-champions-league/2014-2015/2014-11-04 - 22-45 Arsenal 3 - 3 Anderlecht/15/53480-1907-32806-4998-Player_team_right-16-03ar015_229692bb0005-217x150.png', 'train/europe_uefa-champions-league/2014-2015/2014-11-04 - 22-45 Arsenal 3 - 3 Anderlecht/15/53494-1907-32814-5000-Player_team_right-9-03ar015_0295503b0007-144x69.png']
    sample id='2014-11-04 - 22-45 Arsenal 3 - 3 Ande

## 4b) Optional -- MSMT17 pretraining for a stronger starting point

OSNet's default `pretrained=True` weights already come from ImageNet +
Market1501/MSMT17-style pretraining, so this step is **not required** --
skip it if you just want to fine-tune directly on SoccerNet-ReID (Step 5
already does that with solid default weights).

If you want a stronger, larger-scale pedestrian-ReID starting point before
the soccer-specific fine-tune, attach the official MSMT17 dataset (requires
its own registration at https://www.pkuvmc.com/dataset.html -- separate
from SoccerNet's NDA) as a Kaggle Dataset via **Add Data**, and this cell
will pretrain OSNet on it first. If it's not attached, this cell skips
cleanly and Step 5 proceeds with the standard pretrained weights.

In [52]:
# Change this to wherever your attached MSMT17 Kaggle Dataset mounts.
# Expected layout: the official MSMT17 release's own train/test folders
# (torchreid has built-in support for parsing this format directly).
MSMT17_ROOT = Path("/kaggle/input/msmt17")
msmt17_available = MSMT17_ROOT.is_dir() and any(MSMT17_ROOT.iterdir())
msmt17_checkpoint = None

if not msmt17_available:
    print(f"Skipping MSMT17 pretraining -- dataset not found at {MSMT17_ROOT}.\n"
          "This is optional. Step 5 will proceed with OSNet's standard "
          "pretrained weights, which is a perfectly reasonable default. To "
          "use MSMT17 instead, register at https://www.pkuvmc.com/dataset.html, "
          "attach the dataset via 'Add Data', then re-run from this cell.")
else:
    print("MSMT17 dataset found:", MSMT17_ROOT)

Skipping MSMT17 pretraining -- dataset not found at /kaggle/input/msmt17.
This is optional. Step 5 will proceed with OSNet's standard pretrained weights, which is a perfectly reasonable default. To use MSMT17 instead, register at https://www.pkuvmc.com/dataset.html, attach the dataset via 'Add Data', then re-run from this cell.


### 4c) Pretrain OSNet on MSMT17 (only runs if MSMT17 is attached)

In [53]:
if msmt17_available:
    def _pretrain_msmt17(batch_size):
        datamanager = torchreid.data.ImageDataManager(
            root=str(MSMT17_ROOT.parent),
            sources="msmt17",
            height=256,
            width=128,
            batch_size_train=batch_size,
            batch_size_test=batch_size,
            transforms=["random_flip", "color_jitter"],
        )

        pre_model = torchreid.models.build_model(
            name="osnet_x0_25",
            num_classes=datamanager.num_train_pids,
            loss="triplet",
            pretrained=True,  # start from ImageNet weights, then pretrain on MSMT17
        ).cuda()

        optimizer = torchreid.optim.build_optimizer(pre_model, optim="adam", lr=0.0003)
        scheduler = torchreid.optim.build_lr_scheduler(
            optimizer, lr_scheduler="single_step", stepsize=15
        )

        engine = torchreid.engine.ImageTripletEngine(
            datamanager, pre_model, optimizer, margin=0.3, scheduler=scheduler
        )
        engine.run(
            save_dir="/kaggle/working/msmt17_pretrain_log",
            max_epoch=10,  # a short pretrain pass, not a full MSMT17 SOTA run
            eval_freq=10,
            print_freq=50,
            test_only=False,
        )
        return pre_model

    try:
        pretrained_model = _pretrain_msmt17(batch_size=16)
    except torch.cuda.OutOfMemoryError:
        print("CUDA OOM at batch_size=16 -- retrying once at batch_size=4.")
        torch.cuda.empty_cache()
        pretrained_model = _pretrain_msmt17(batch_size=4)

    import glob as _glob
    _ckpts = sorted(_glob.glob("/kaggle/working/msmt17_pretrain_log/model/model.pth.tar-*"))
    if _ckpts:
        msmt17_checkpoint = _ckpts[-1]
        print(f"MSMT17 pretraining done. Checkpoint: {msmt17_checkpoint}")
    else:
        print("MSMT17 pretraining ran but no checkpoint was found -- Step 5 "
              "will fall back to standard pretrained weights.")

## 5) Train OSNet-x0.25 (fine-tune from ImageNet-pretrained weights)

Fine-tuning rather than training from scratch -- SoccerNet-ReID is almost
certainly far smaller than the ImageNet-scale ReID benchmarks OSNet was
designed for, so starting from pretrained weights is much more
sample-efficient. Includes an OOM-fallback retry at a smaller batch size.

In [54]:
import torchreid

# register_image_dataset isn't idempotent -- guard it so re-running this
# cell in the same kernel session (e.g. after fixing an error further
# down) doesn't raise "name already exists".
if "soccernet_reid" not in torchreid.data.datasets.__image_datasets:
    torchreid.data.register_image_dataset(
        "soccernet_reid",
        lambda **kwargs: torchreid.data.datasets.image.market1501.Market1501(
            root=str(REID_ROOT.parent), **kwargs
        ),
    )
else:
    print("'soccernet_reid' already registered in this session -- skipping.")
# torchreid's Market1501 loader expects the dataset dir to be named
# 'market1501'; alias our folder so it's discovered without copying data again.
alias = REID_ROOT.parent / "market1501"
if not alias.exists():
    alias.symlink_to(REID_ROOT, target_is_directory=True)


def _build_and_train(batch_size):
    datamanager = torchreid.data.ImageDataManager(
        root=str(REID_ROOT.parent),
        sources="market1501",
        height=256,
        width=128,
        batch_size_train=batch_size,
        batch_size_test=batch_size,
        transforms=["random_flip", "color_jitter"],
    )

    model = torchreid.models.build_model(
        name="osnet_x0_25",
        num_classes=datamanager.num_train_pids,
        loss="triplet",
        pretrained=(msmt17_checkpoint is None),  # skip ImageNet-only weights if we have an MSMT17 checkpoint
    ).cuda()

    if msmt17_checkpoint is not None:
        torchreid.utils.load_pretrained_weights(model, msmt17_checkpoint)
        print(f"Loaded MSMT17-pretrained weights from {msmt17_checkpoint} "
              "as the starting point for SoccerNet-ReID fine-tuning.")

    optimizer = torchreid.optim.build_optimizer(model, optim="adam", lr=0.0003)
    scheduler = torchreid.optim.build_lr_scheduler(
        optimizer, lr_scheduler="single_step", stepsize=20
    )

    engine = torchreid.engine.ImageTripletEngine(
        datamanager, model, optimizer, margin=0.3, scheduler=scheduler
    )

    engine.run(
        save_dir="/kaggle/working/reid_log",
        max_epoch=40,  # raised from 20 -- loss was still dropping meaningfully at 20
        eval_freq=5,
        print_freq=20,
        test_only=False,
    )
    return engine, model


try:
    engine, model = _build_and_train(batch_size=16)
except torch.cuda.OutOfMemoryError:
    print("CUDA OOM at batch_size=16 -- retrying once at batch_size=4.")
    torch.cuda.empty_cache()
    engine, model = _build_and_train(batch_size=4)

'soccernet_reid' already registered in this session -- skipping.
Building train transforms ...
+ resize to 256x128
+ random flip
+ color jitter
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
Building test transforms ...
+ resize to 256x128
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |  1200 |     9463 |         6
  query    |  1200 |     1200 |         1
  gallery  |  1200 |     8283 |         6
  ----------------------------------------
=> Loading test (target) dataset
=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |  1200 |     9463 |         6
  query    |  1200 

## 6) Evaluate -- rank-1 / mAP on the held-out split

In [55]:
engine.run(
    save_dir="/kaggle/working/reid_log",
    test_only=True,
    visrank=False,
)

##### Evaluating market1501 (source) #####
Extracting features from query set ...
Done, obtained 1200-by-512 matrix
Extracting features from gallery set ...
Done, obtained 8283-by-512 matrix
Speed: 0.0352 sec/batch
Computing distance matrix with metric=euclidean ...
Computing CMC and mAP ...
** Results **
mAP: 8.4%
CMC curve
Rank-1  : 21.8%
Rank-5  : 38.9%
Rank-10 : 48.1%
Rank-20 : 57.3%


## 7) Export best checkpoint for download

In [56]:
import glob

checkpoints = sorted(glob.glob("/kaggle/working/reid_log/model/model.pth.tar-*"))
assert checkpoints, "No checkpoint files found under /kaggle/working/reid_log/model/"
best_checkpoint = checkpoints[-1]  # highest epoch number

shutil.copy(best_checkpoint, "/kaggle/working/best_reid.pt")
shutil.make_archive(
    "/kaggle/working/reid_osnet_outputs", "zip", "/kaggle/working/reid_log"
)

print("Ready to download from the Kaggle Output panel:")
print("  /kaggle/working/best_reid.pt")
print("  /kaggle/working/reid_osnet_outputs.zip")

Ready to download from the Kaggle Output panel:
  /kaggle/working/best_reid.pt
  /kaggle/working/reid_osnet_outputs.zip
